# ALQAC 2026 — K23 submission run (Colab, A100)

Runs the full pipeline: case-evidence retrieval (Case Content API, with cache) +
outcome ensemble (self-consistency 5 + precedents + thinking) + law retrieval
(citation extraction + procedural priors), then writes `submission.json` and the
organizer-format file `[submission] K23.json`.

The `ensemble.yaml` config sets `use_precedents=true`, which reads the labelled
public cases as an inference-time retrieval memory (permitted by the competition
rules, since the public set is the only labelled signal). To disable it, set
`use_precedents: false` in `configs/ensemble.yaml`.

## Preparation
1. Runtime → **A100**, Internet On.
2. Upload (via Files or Google Drive):
   - `alqac2026_kaggle_bundle.zip` (the packaged source bundle)
   - `case_api.sqlite` (the Case Content API cache, first run only)
3. Provide the API token as a Colab Secret named `ALQAC_TEAM_TOKEN`, or paste it
   into `MY_TOKEN` in the setup cell.


## 1. Setup: unpack the bundle, install dependencies, load the token, restore the cache

In [ ]:
import os, sys, json, zipfile, shutil, subprocess
from pathlib import Path

# Mount Google Drive so the cache and checkpoints survive a disconnect (required for resume).
from google.colab import drive
drive.mount('/content/drive')
PERSIST = Path('/content/drive/MyDrive/ALQAC_RUN')       # persistent working directory
(PERSIST/'cache').mkdir(parents=True, exist_ok=True)
(PERSIST/'outputs').mkdir(parents=True, exist_ok=True)

def find(name):
    for root in ('/content','/content/drive','.'):
        rp=Path(root)
        if rp.exists():
            hits=sorted(rp.rglob(name), key=lambda q:-q.stat().st_size)
            if hits: return hits[0]
    return None

# API token: prefer a Colab Secret, otherwise paste it into MY_TOKEN below.
MY_TOKEN = ''
try:
    from google.colab import userdata
    tok = userdata.get('ALQAC_TEAM_TOKEN')
    if tok: os.environ['ALQAC_TEAM_TOKEN']=tok.strip()
except Exception: pass
if MY_TOKEN: os.environ['ALQAC_TEAM_TOKEN']=MY_TOKEN.strip()
assert os.environ.get('ALQAC_TEAM_TOKEN'), 'Missing ALQAC_TEAM_TOKEN (set a Colab Secret or MY_TOKEN)'

# Unpack the source bundle.
PROJECT_ROOT=Path('/content/DL_K23')
z=find('alqac2026_kaggle_bundle.zip') or find('*bundle*.zip')
assert z, 'Upload alqac2026_kaggle_bundle.zip'
shutil.rmtree('/content/_x', ignore_errors=True)
with zipfile.ZipFile(z) as zf: zf.extractall('/content/_x')
src=next(pp.parent for pp in Path('/content/_x').rglob('pyproject.toml') if (pp.parent/'src'/'alqac2026').exists())
if PROJECT_ROOT.exists(): shutil.rmtree(PROJECT_ROOT)
shutil.copytree(src, PROJECT_ROOT)

# Install dependencies.
subprocess.run([sys.executable,'-m','pip','install','-q','transformers>=4.51','accelerate',
                'bitsandbytes','sentence-transformers','rank-bm25','pyvi','requests','pyyaml',
                'python-dotenv','tqdm'], check=False)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(PROJECT_ROOT),'--no-deps'], check=True)

# Keep the API cache on Drive so it persists across sessions.
CACHE_DB = PERSIST/'cache/case_api.sqlite'
if not CACHE_DB.exists():
    up = find('case_api.sqlite')   # seed from the uploaded cache on the first run
    if up and str(up)!=str(CACHE_DB):
        shutil.copy2(up, CACHE_DB); print('cache seeded from', up)
# Point cache_db and law_index at Drive so both survive a disconnect.
import yaml
for cfgname in ('ensemble.yaml','baseline.yaml','candidate.yaml'):
    cfgp=PROJECT_ROOT/'configs'/cfgname
    if cfgp.exists():
        y=yaml.safe_load(cfgp.read_text(encoding='utf-8'))
        y['paths']['cache_db']=str(CACHE_DB)
        y['paths']['law_index']=str(PERSIST/'law_index')
        cfgp.write_text(yaml.safe_dump(y, allow_unicode=True, sort_keys=False), encoding='utf-8')
print('cache_db =', CACHE_DB, '| exists:', CACHE_DB.exists())
import torch; print('GPU:', torch.cuda.get_device_name(0))

## 2. Select the phase (run `public` to confirm the format; run `private` only once, after public is verified)

In [ ]:
PHASE   = 'public'    # public | private
RUN_MODE= 'full'      # smoke (2 cases) | full
assert PHASE in {'public','private'} and RUN_MODE in {'smoke','full'}
LIMIT = 2 if RUN_MODE=='smoke' else None
RUN_DIR = PERSIST/'outputs'/f'{PHASE}_ensemble_{RUN_MODE}'   # on Drive so the run can resume

if PHASE=='public':
    INPUT='data/raw/ALQAC2026_public_test.json'; RUNNER='scripts/run_public.py'
else:
    # Accept only the file that contains case_query; avoid picking the law corpus by mistake.
    def find_private_cases():
        for base in ('/content','/content/drive','.'):
            b=Path(base)
            if not b.exists(): continue
            for q in b.rglob('*.json'):
                try:
                    d=json.loads(q.read_text(encoding='utf-8'))
                    if isinstance(d,list) and d and 'case_query' in d[0] and 'case_id' in d[0] and len(d)<=200:
                        return q
                except Exception: pass
        return None
    pv=find_private_cases()
    assert pv, 'Upload the private test file (cases with case_id + case_query)'
    INPUT=str(pv); RUNNER='scripts/run_private.py'
    print('Private cases:', len(json.loads(Path(pv).read_text(encoding="utf-8"))))
    # The organizers release a separate law corpus for the private phase; use it for law retrieval.
    def find_private_corpus():
        for base in ('/content','/content/drive','.'):
            b=Path(base)
            if not b.exists(): continue
            for q in b.rglob('*.json'):
                try:
                    d=json.loads(q.read_text(encoding='utf-8'))
                    if isinstance(d,list) and d and 'law_id' in d[0] and 'content' in d[0]:
                        return q
                except Exception: pass
        return None
    pc=find_private_corpus()
    if pc:
        import yaml
        cfgp=PROJECT_ROOT/'configs/ensemble.yaml'
        y=yaml.safe_load(cfgp.read_text(encoding='utf-8')); y['paths']['corpus']=str(pc)
        cfgp.write_text(yaml.safe_dump(y, allow_unicode=True, sort_keys=False), encoding='utf-8')
        print('Using private corpus:', pc)
    else:
        print('!! Private corpus (law_id + content) not found -> falling back to the public corpus. Upload the private corpus.')

    print('PRIVATE input:', INPUT, '| *** run exactly once ***')
print({'phase':PHASE,'mode':RUN_MODE,'input':INPUT,'run_dir':str(RUN_DIR)})

## 3. Run the pipeline (retrieval + outcome ensemble + law). If it disconnects, re-run this cell to resume.

In [ ]:
cmd=[sys.executable, str(PROJECT_ROOT/RUNNER),
     '--config','configs/ensemble.yaml','--input',INPUT,'--resume-run',str(RUN_DIR)]
if LIMIT is not None: cmd += ['--limit',str(LIMIT)]
proc=subprocess.run(cmd, cwd=PROJECT_ROOT, env=os.environ.copy(), check=False)
if proc.returncode!=0:
    for fn in ('manifest.json','api_stats.json'):
        f=RUN_DIR/fn
        if f.exists(): print(fn, json.loads(f.read_text(encoding='utf-8')).get('run'))
    raise SystemExit(f'FAILED rc={proc.returncode}')
print('Pipeline OK')

## 4. Results and the submission file (`[submission] K23.json`, in the organizer format)

In [ ]:
def rj(p): return json.loads(Path(p).read_text(encoding='utf-8'))
val=rj(RUN_DIR/'validation.json'); print('VALIDATION:', val)
if (RUN_DIR/'metrics.json').exists():
    print('METRICS:', json.dumps(rj(RUN_DIR/'metrics.json'), ensure_ascii=False, indent=2))
print('API STATS:', json.dumps(rj(RUN_DIR/'api_stats.json'), ensure_ascii=False))

GROUP='K23'
sub=rj(RUN_DIR/'submission.json')
out=Path('/content')/f'[submission] {GROUP}.json'
out.write_text(json.dumps(sub, ensure_ascii=False), encoding='utf-8')
print(f'\nSUBMISSION FILE: {out}  ({len(sub)} cases)')
print('Download it from the Files panel (left). Source submission.json:', RUN_DIR/'submission.json')